In [14]:
import matplotlib.pyplot as plt
import numpy as np 
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
class MLP_Classifier:
    """
    Description:
        Multi-Layer Perceptron (MLP) for Multi-class Classification.
        Uses RELU for hidden layers and Softmax for the output layer.
    """
    def __init__(self, layer_sizes=[64], output_size=3, learning_rate=0.01):
        self.layer_sizes = layer_sizes
        self.output_size = output_size
        self.learning_rate = learning_rate
        self.weights = []
        self.biases = []
        self.zs = []
        self.activations = []
        
        if len(layer_sizes) < 1:
            raise ValueError("At least one hidden layer is required.")
    
    def _initialize_weights(self, input_size):
        all_layers = [input_size] + self.layer_sizes + [self.output_size]
        for i in range(len(all_layers) - 1):
            self.weights.append(np.random.randn(all_layers[i], all_layers[i+1]) * 0.01)
            self.biases.append(np.zeros((1, all_layers[i+1])))

    def _softmax(self, z):
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)
    
    def _relu(self, z):
        return np.maximum(0, z)
    
    def _relu_derivative(self, z):
        return (z > 0).astype(float)
    
    def _forward(self, X):
        self.zs = []
        self.activations = [X]
        for i in range(len(self.weights)):
            z = np.dot(self.activations[-1], self.weights[i]) + self.biases[i]
            self.zs.append(z)
            
            if i < len(self.weights) - 1:
                a = self._relu(z)
            else:
                a = self._softmax(z)

            self.activations.append(a)
        return self.activations[-1]
    
    def _backward(self, y_true):
        m = y_true.shape[0]
        dZ = self.activations[-1] - y_true
        
        for i in reversed(range(len(self.weights))):
            A_prev = self.activations[i]
            dW = np.dot(A_prev.T, dZ) / m
            db = np.sum(dZ, axis=0, keepdims=True) / m
            
            if i > 0:
                dA_prev = np.dot(dZ, self.weights[i].T)
                dZ = dA_prev * self._relu_derivative(self.zs[i-1])
                
            self.weights[i] -= self.learning_rate * dW
            self.biases[i] -= self.learning_rate * db

    def fit(self, X, y, epochs=10, batch_size=32):
        """
        Huấn luyện mô hình sử dụng Mini-batch Stochastic Gradient Descent.
        Args:
            X: Dữ liệu huấn luyện (shape: m, input_size)
            y: Nhãn dạng One-Hot Encoded (shape: m, output_size)
            epochs: Số lần duyệt qua toàn bộ tập dữ liệu
            batch_size: Số lượng mẫu trong mỗi mini-batch
        """
        if not self.weights:
            self._initialize_weights(X.shape[1])
            
        m = X.shape[0] # Tổng số lượng mẫu dữ liệu
        
        for epoch in range(epochs):
            # 1. Xáo trộn dữ liệu (Shuffle) ở đầu mỗi epoch
            permutation = np.random.permutation(m)
            X_shuffled = X[permutation]
            y_shuffled = y[permutation]
            
            epoch_loss = 0
            
            # 2. Chia dữ liệu thành các mini-batch
            for i in range(0, m, batch_size):
                # Cắt lấy một batch
                X_batch = X_shuffled[i : i + batch_size]
                y_batch = y_shuffled[i : i + batch_size]
                
                # 3. Lan truyền tiến và lùi trên TỪNG batch
                self._forward(X_batch)
                self._backward(y_batch)
                
                # Tính loss cho batch hiện tại để cộng dồn
                predictions = self.activations[-1]
                batch_loss = -np.mean(np.sum(y_batch * np.log(predictions + 1e-8), axis=1))
                
                # Nhân với số lượng mẫu thực tế của batch (phòng trường hợp batch cuối bị lẻ)
                epoch_loss += batch_loss * X_batch.shape[0]
            
            # Tính loss trung bình của toàn bộ epoch
            epoch_loss /= m
            
            # Do dùng mini-batch, số epochs thường nhỏ hơn nhiều nên ta có thể in ra mỗi epoch
            print(f"Epoch {epoch + 1}/{epochs}, Loss: {epoch_loss:.4f}")

    def predict(self, X):
        probabilities = self._forward(X)
        return np.argmax(probabilities, axis=1)

In [16]:
transform = transforms.ToTensor()

# Tải tập huấn luyện (Train)
print("Đang tải tập Train...")
train_dataset = datasets.MNIST(root='../data',     
                               train=True,          
                               download=True,      
                               transform=transform)
# Tải tập kiểm tra (Test)
print("Đang tải tập Test...")
test_dataset = datasets.MNIST(root='../data', 
                              train=False,    
                              download=True, 
                              transform=transform)
print(f"Hoàn tất! Số lượng ảnh train: {len(train_dataset)}")
print(f"Hoàn tất! Số lượng ảnh test: {len(test_dataset)}")
print(f"Kích thước ảnh train: {train_dataset.data.shape}")
print(f"Kích thước ảnh test: {test_dataset.data.shape}")

Đang tải tập Train...
Đang tải tập Test...
Hoàn tất! Số lượng ảnh train: 60000
Hoàn tất! Số lượng ảnh test: 10000
Kích thước ảnh train: torch.Size([60000, 28, 28])
Kích thước ảnh test: torch.Size([10000, 28, 28])


In [ ]:
X_train = train_dataset.data.numpy().reshape(-1, 28*28) / 255.0
X_test = test_dataset.data.numpy().reshape(-1, 28*28) / 255.0

y_train_labels = train_dataset.targets.numpy()
y_test_labels = test_dataset.targets.numpy()

def one_hot_encode(y, num_classes=10):
    m = y.shape[0]
    one_hot = np.zeros((m, num_classes))
    one_hot[np.arange(m), y] = 1
    return one_hot

y_train_one_hot = one_hot_encode(y_train_labels, num_classes=10)

model = MLP_Classifier(layer_sizes=[128, 64], output_size=10, learning_rate=0.1)

print("\nBắt đầu quá trình huấn luyện (Training)...")
model.fit(X_train, y_train_one_hot, epochs=10, batch_size=64)

print("\nĐang dự đoán trên tập Test...")
predictions = model.predict(X_test)

accuracy = np.mean(predictions == y_test_labels)
print(f"Độ chính xác (Accuracy): {accuracy * 100:.2f}%")


Bắt đầu quá trình huấn luyện (Training)...
